In [3]:
from pathlib import Path
import xml.etree.ElementTree as ET
from collections import Counter

# Dataset root
DATASET_ROOT = Path(r"C:\major-project\model\dataset\apple-dataset")

# Actual nested folders
ANNOTATIONS_DIR = DATASET_ROOT / "Annotations" / "Annotations"
IMAGES_DIR = DATASET_ROOT / "JPEGImages" / "JPEGImages"

print("Annotations path:", ANNOTATIONS_DIR)
print("Images path:", IMAGES_DIR)

print("Annotations exists:", ANNOTATIONS_DIR.exists())
print("Images exists:", IMAGES_DIR.exists())

# Count files
xml_files = list(ANNOTATIONS_DIR.glob("*.xml"))
jpg_files = list(IMAGES_DIR.glob("*.jpg"))

print("\nXML annotation files:", len(xml_files))
print("JPG image files:", len(jpg_files))

# Find classes
classes = Counter()

for xml_file in xml_files:
    try:
        root = ET.parse(xml_file).getroot()

        for obj in root.findall("object"):
            class_name = obj.findtext("name", "").strip()
            if class_name:
                classes[class_name] += 1
    except Exception as e:
        print(f"Error reading {xml_file.name}: {e}")

print("\nClasses found:")
for name, count in classes.items():
    print(f"  {name}: {count} objects")

Annotations path: C:\major-project\model\dataset\apple-dataset\Annotations\Annotations
Images path: C:\major-project\model\dataset\apple-dataset\JPEGImages\JPEGImages
Annotations exists: True
Images exists: True

XML annotation files: 4733
JPG image files: 4733

Classes found:
  apple: 29160 objects


In [4]:
from pathlib import Path
import xml.etree.ElementTree as ET
import shutil
import random

# ============================================================
# PATHS
# ============================================================

SOURCE_ROOT = Path(r"C:\major-project\model\dataset\apple-dataset")

SOURCE_IMAGES = SOURCE_ROOT / "JPEGImages" / "JPEGImages"
SOURCE_ANNOTATIONS = SOURCE_ROOT / "Annotations" / "Annotations"

# New YOLO-ready dataset location
YOLO_DATASET = Path(r"C:\major-project\model\apple_yolo_dataset")

# Create output folders
for split in ["train", "val", "test"]:
    (YOLO_DATASET / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DATASET / "labels" / split).mkdir(parents=True, exist_ok=True)

# ============================================================
# XML -> YOLO CONVERSION FUNCTION
# ============================================================

def convert_bbox(size, box):
    """
    Convert Pascal VOC bounding box to YOLO format.
    YOLO format: class x_center y_center width height
    All coordinates are normalized between 0 and 1.
    """
    img_width, img_height = size

    xmin, ymin, xmax, ymax = box

    x_center = ((xmin + xmax) / 2) / img_width
    y_center = ((ymin + ymax) / 2) / img_height
    width = (xmax - xmin) / img_width
    height = (ymax - ymin) / img_height

    # Keep values inside valid YOLO range
    x_center = min(max(x_center, 0), 1)
    y_center = min(max(y_center, 0), 1)
    width = min(max(width, 0), 1)
    height = min(max(height, 0), 1)

    return x_center, y_center, width, height


# ============================================================
# GET VALID IMAGE/XML PAIRS
# ============================================================

image_files = list(SOURCE_IMAGES.glob("*.jpg"))

pairs = []

for image_path in image_files:
    xml_path = SOURCE_ANNOTATIONS / f"{image_path.stem}.xml"

    if xml_path.exists():
        pairs.append((image_path, xml_path))

print(f"Valid image/XML pairs found: {len(pairs)}")

# ============================================================
# SPLIT DATASET: 80% TRAIN, 10% VALIDATION, 10% TEST
# ============================================================

random.seed(42)
random.shuffle(pairs)

total = len(pairs)
train_end = int(total * 0.80)
val_end = int(total * 0.90)

splits = {
    "train": pairs[:train_end],
    "val": pairs[train_end:val_end],
    "test": pairs[val_end:]
}

print("\nDataset split:")
for split_name, split_pairs in splits.items():
    print(f"{split_name}: {len(split_pairs)} images")

# ============================================================
# CONVERT AND COPY DATA
# ============================================================

conversion_stats = {
    "images_processed": 0,
    "apple_annotations": 0,
    "skipped_annotations": 0
}

for split_name, split_pairs in splits.items():

    for image_path, xml_path in split_pairs:

        try:
            root = ET.parse(xml_path).getroot()

            # Get image dimensions from XML
            size = root.find("size")
            img_width = float(size.find("width").text)
            img_height = float(size.find("height").text)

            yolo_lines = []

            # Process every apple object
            for obj in root.findall("object"):

                class_name = obj.findtext("name", "").strip().lower()

                # This is an APPLE-ONLY dataset/model
                if class_name != "apple":
                    conversion_stats["skipped_annotations"] += 1
                    continue

                difficult = obj.findtext("difficult", "0")
                if difficult == "1":
                    # We can still include difficult objects, so don't skip them
                    pass

                bbox = obj.find("bndbox")
                if bbox is None:
                    conversion_stats["skipped_annotations"] += 1
                    continue

                xmin = float(bbox.find("xmin").text)
                ymin = float(bbox.find("ymin").text)
                xmax = float(bbox.find("xmax").text)
                ymax = float(bbox.find("ymax").text)

                # Validate bounding box
                if xmax <= xmin or ymax <= ymin:
                    conversion_stats["skipped_annotations"] += 1
                    continue

                x, y, w, h = convert_bbox(
                    (img_width, img_height),
                    (xmin, ymin, xmax, ymax)
                )

                # YOLO class 0 = apple
                yolo_lines.append(
                    f"0 {x:.6f} {y:.6f} {w:.6f} {h:.6f}"
                )

                conversion_stats["apple_annotations"] += 1

            # Copy image
            destination_image = (
                YOLO_DATASET / "images" / split_name / image_path.name
            )
            shutil.copy2(image_path, destination_image)

            # Save YOLO label
            destination_label = (
                YOLO_DATASET / "labels" / split_name /
                f"{image_path.stem}.txt"
            )

            with open(destination_label, "w", encoding="utf-8") as f:
                f.write("\n".join(yolo_lines))

            conversion_stats["images_processed"] += 1

        except Exception as e:
            print(f"Error processing {image_path.name}: {e}")

print("\n========== CONVERSION COMPLETED ==========")
print(f"Images processed: {conversion_stats['images_processed']}")
print(f"Apple annotations converted: {conversion_stats['apple_annotations']}")
print(f"Skipped annotations: {conversion_stats['skipped_annotations']}")
print(f"\nYOLO dataset saved at:\n{YOLO_DATASET}")

Valid image/XML pairs found: 4733

Dataset split:
train: 3786 images
val: 473 images
test: 474 images

========== CONVERSION COMPLETED ==========
Images processed: 4733
Apple annotations converted: 29160
Skipped annotations: 0

YOLO dataset saved at:
C:\major-project\model\apple_yolo_dataset


In [5]:
from pathlib import Path
from collections import Counter

YOLO_DATASET = Path(r"C:\major-project\model\apple_yolo_dataset")

print("========== YOLO DATASET VERIFICATION ==========")

for split in ["train", "val", "test"]:
    images_dir = YOLO_DATASET / "images" / split
    labels_dir = YOLO_DATASET / "labels" / split

    images = list(images_dir.glob("*.jpg"))
    labels = list(labels_dir.glob("*.txt"))

    total_objects = 0
    class_counts = Counter()
    empty_labels = 0

    for label_file in labels:
        content = label_file.read_text(encoding="utf-8").strip()

        if not content:
            empty_labels += 1
            continue

        for line in content.splitlines():
            parts = line.split()

            if len(parts) == 5:
                class_id = parts[0]
                class_counts[class_id] += 1
                total_objects += 1

    print(f"\n{split.upper()}")
    print(f"Images: {len(images)}")
    print(f"Labels: {len(labels)}")
    print(f"Apple objects: {total_objects}")
    print(f"Empty labels: {empty_labels}")
    print(f"Classes found: {dict(class_counts)}")

========== YOLO DATASET VERIFICATION ==========

TRAIN
Images: 3786
Labels: 3786
Apple objects: 23281
Empty labels: 0
Classes found: {'0': 23281}

VAL
Images: 473
Labels: 473
Apple objects: 2891
Empty labels: 0
Classes found: {'0': 2891}

TEST
Images: 474
Labels: 474
Apple objects: 2988
Empty labels: 0
Classes found: {'0': 2988}


In [ ]:
from pathlib import Path

YOLO_DATASET = Path(r"C:\major-project\model\apple_yolo_dataset")

DATA_YAML = YOLO_DATASET / "data.yaml"

yaml_content = f"""
path: {YOLO_DATASET.as_posix()}
train: images/train
val: images/val
test: images/test

names:
  0: apple
"""

DATA_YAML.write_text(yaml_content.strip(), encoding="utf-8")

print("data.yaml created successfully!")
print("\nLocation:", DATA_YAML)
print("\nContent:\n")
print(DATA_YAML.read_text())

data.yaml created successfully!

Location: C:\major-project\model\apple_yolo_dataset\data.yaml

Content:

path: C:/major-project/model/apple_yolo_dataset
train: images/train
val: images/val
test: images/test

names:
  0: apple


In [7]:
from ultralytics import YOLO

# Dataset configuration
DATA_YAML = r"C:\major-project\model\apple_yolo_dataset\data.yaml"

# IMPORTANT:
# Use the local pretrained model path if you already have it.
MODEL_PATH = r"C:\major-project\model\yolo11\yolo11n.pt"

# Load pretrained YOLO11 Nano model
model = YOLO(MODEL_PATH)

# Train Apple-only detection model
results = model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=8,
    device=0,          # RTX 2050 GPU
    patience=20,
    project=r"C:\major-project\model\apple_models",
    name="apple_detection_100epochs",
    exist_ok=True,
    plots=True
)

print("\nApple-only training completed!")

c:\Users\kiran\miniconda3\envs\venv_rating\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.5.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


Ultralytics 8.4.126  Python-3.10.18 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 2050, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\major-project\model\apple_yolo_dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=C:\major-project\model\yolo11\yolo11n.pt, momentum=0.937, m

KeyboardInterrupt: 

In [8]:
from ultralytics import YOLO

BEST_MODEL = r"C:\major-project\model\yolo11\runs\apple_detection_gpu\weights\best.pt"
DATA_YAML = r"C:\major-project\model\apple_yolo_dataset\data.yaml"

# Load the best model
best_model = YOLO(BEST_MODEL)

# Evaluate on the separate TEST dataset
test_results = best_model.val(
    data=DATA_YAML,
    split="test",
    imgsz=640,
    device=0
)

print("\n========== FINAL APPLE DETECTION TEST RESULTS ==========")
print(f"Precision:  {test_results.box.mp:.2%}")
print(f"Recall:     {test_results.box.mr:.2%}")
print(f"mAP@50:     {test_results.box.map50:.2%}")
print(f"mAP@50-95:  {test_results.box.map:.2%}")

Ultralytics 8.4.126  Python-3.10.18 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 2050, 4096MiB)
Model summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
WARNING val: Slow image access detected (ping: 0.60.4 ms, read: 9.55.6 MB/s, size: 69.8 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning C:\major-project\model\apple_yolo_dataset\labels\test... 474 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 474/474 662.7it/s 0.7s0.0s
val: New cache created: C:\major-project\model\apple_yolo_dataset\labels\test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 0% ──────────── 0/30  1.6s


error: Caught error in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "c:\Users\kiran\miniconda3\envs\venv_rating\lib\site-packages\torch\utils\data\_utils\worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
  File "c:\Users\kiran\miniconda3\envs\venv_rating\lib\site-packages\torch\utils\data\_utils\fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
  File "c:\Users\kiran\miniconda3\envs\venv_rating\lib\site-packages\torch\utils\data\_utils\fetch.py", line 54, in <listcomp>
    data = [self.dataset[idx] for idx in possibly_batched_index]
  File "c:\Users\kiran\miniconda3\envs\venv_rating\lib\site-packages\ultralytics\data\base.py", line 397, in __getitem__
    return self.transforms(self.get_image_and_label(index))
  File "c:\Users\kiran\miniconda3\envs\venv_rating\lib\site-packages\ultralytics\data\base.py", line 410, in get_image_and_label
    label["img"]

In [ ]:
from ultralytics import YOLO

BEST_MODEL = r"C:\major-project\model\yolo11\runs\apple_detection_gpu\weights\best.pt"
DATA_YAML = r"C:\major-project\model\apple_yolo_dataset\data.yaml"

# Load the best model
best_model = YOLO(BEST_MODEL)

# Evaluate on the separate TEST dataset
test_results = best_model.val(
    data=DATA_YAML,
    split="test",
    imgsz=640,
    device=0,
    batch=4,      # smaller batch to reduce memory usage
    workers=0     # important: prevents RAM issue
)

print("\n========== FINAL APPLE DETECTION TEST RESULTS ==========")
print(f"Precision:  {test_results.box.mp:.2%}")
print(f"Recall:     {test_results.box.mr:.2%}")
print(f"mAP@50:     {test_results.box.map50:.2%}")
print(f"mAP@50-95:  {test_results.box.map:.2%}")

Ultralytics 8.4.126  Python-3.10.18 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 2050, 4096MiB)
Model summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 102.478.4 MB/s, size: 38.8 KB)
val: Scanning C:\major-project\model\apple_yolo_dataset\labels\test.cache... 474 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 474/474  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 119/119 10.3it/s 11.6s0.2s
                   all        474       2988      0.566      0.463      0.512      0.372
Speed: 0.4ms preprocess, 13.4ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to C:\Users\kiran\runs\detect\val-10

========== FINAL APPLE DETECTION TEST RESULTS ==========
Precision:  56.62%
Recall:     46.32%
mAP@50:     51.22%
mAP@50-95:  37.17%


In [1]:
from ultralytics import YOLO
from pathlib import Path
import uuid
import shutil
from datetime import datetime

# ============================================================
# BATCH-WISE APPLE DETECTION CONFIGURATION
# ============================================================

MODEL_PATH = Path(r"C:\major-project\model\yolo11\runs\apple_detection_gpu\weights\best.pt")

# Folder containing multiple images to process as one batch
INPUT_FOLDER = Path(r"C:\major-project\model\batch_input")

# Folder where batch results will be stored
OUTPUT_FOLDER = Path(r"C:\major-project\model\batch_results")

# Create folders if they don't exist
INPUT_FOLDER.mkdir(parents=True, exist_ok=True)
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

# Load the trained Apple detection model
batch_model = YOLO(str(MODEL_PATH))

print("Batch processing system initialized successfully!")
print("Model:", MODEL_PATH)
print("Input folder:", INPUT_FOLDER)
print("Output folder:", OUTPUT_FOLDER)

Batch processing system initialized successfully!
Model: C:\major-project\model\yolo11\runs\apple_detection_gpu\weights\best.pt
Input folder: C:\major-project\model\batch_input
Output folder: C:\major-project\model\batch_results


In [2]:
from pathlib import Path
from ultralytics import YOLO

# ============================================================
# BATCH PROCESSING CONFIGURATION
# ============================================================

# Your trained Apple detection model
MODEL_PATH = r"C:\major-project\model\yolo11\runs\apple_detection_gpu\weights\best.pt"

# Your specific batch folder containing multiple images
INPUT_FOLDER = Path(r"C:\major-project\model\batch_input\batch1")

# Main folder where results will be saved
OUTPUT_FOLDER = Path(r"C:\major-project\model\batch_results")

# Create output folder if it doesn't exist
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

# Load the trained model
batch_model = YOLO(MODEL_PATH)

# Check everything
print("✅ Batch processing system initialized!")
print(f"🤖 Model: {MODEL_PATH}")
print(f"📥 Input folder: {INPUT_FOLDER}")
print(f"📤 Output folder: {OUTPUT_FOLDER}")
print(f"📁 Input folder exists: {INPUT_FOLDER.exists()}")

✅ Batch processing system initialized!
🤖 Model: C:\major-project\model\yolo11\runs\apple_detection_gpu\weights\best.pt
📥 Input folder: C:\major-project\model\batch_input\batch1
📤 Output folder: C:\major-project\model\batch_results
📁 Input folder exists: True


In [ ]:
# ============================================================
# PROCESS BATCH 1 - MULTIPLE IMAGES, MULTIPLE APPLES
# ============================================================

import json
import cv2
from datetime import datetime
import uuid

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Get all images inside batch1
image_paths = sorted([
    file for file in INPUT_FOLDER.iterdir()
    if file.is_file() and file.suffix.lower() in IMAGE_EXTENSIONS
])

print(f"📸 Images found in batch: {len(image_paths)}")

if not image_paths:
    print(f"❌ No images found inside: {INPUT_FOLDER}")

else:
    # Create a unique ID for this processing batch
    batch_id = f"BATCH_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:6]}"
    
    # Create folders for this batch's results
    batch_folder = OUTPUT_FOLDER / batch_id
    annotated_folder = batch_folder / "annotated_images"
    annotated_folder.mkdir(parents=True, exist_ok=True)

    print(f"\n📦 Processing Batch: {batch_id}")
    print("🚀 Starting Apple Detection...\n")

    # Run prediction on all images
    results = batch_model.predict(
        source=[str(path) for path in image_paths],
        conf=0.25,
        device=0,
        verbose=False
    )

    batch_images = []
    total_apples = 0

    # Process each image result
    for image_path, result in zip(image_paths, results):

        # Count all detected apples in this image
        apple_count = len(result.boxes) if result.boxes is not None else 0

        # Get confidence scores
        confidences = []
        if result.boxes is not None and len(result.boxes) > 0:
            confidences = [
                round(float(conf), 4)
                for conf in result.boxes.conf.cpu().tolist()
            ]

        # Save image with detection bounding boxes
        annotated_image = result.plot()
        output_image_path = annotated_folder / image_path.name
        cv2.imwrite(str(output_image_path), annotated_image)

        # Store individual image results
        image_data = {
            "image_name": image_path.name,
            "apple_count": apple_count,
            "average_confidence": round(
                sum(confidences) / len(confidences), 4
            ) if confidences else 0
        }

        batch_images.append(image_data)
        total_apples += apple_count

        print(
            f"✓ {image_path.name} → "
            f"{apple_count} apple(s) detected"
        )

    # Complete batch summary
    batch_summary = {
        "batch_id": batch_id,
        "processed_at": datetime.now().isoformat(),
        "total_images": len(image_paths),
        "total_apples_detected": total_apples,
        "average_apples_per_image": round(
            total_apples / len(image_paths), 2
        ),
        "images": batch_images
    }

    # Save the batch report
    summary_path = batch_folder / "batch_summary.json"

    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(batch_summary, f, indent=4)

    print("\n" + "=" * 55)
    print("✅ BATCH PROCESSING COMPLETED!")
    print("=" * 55)
    print(f"📦 Batch ID: {batch_id}")
    print(f"📸 Total Images Processed: {len(image_paths)}")
    print(f"🍎 Total Apples Detected: {total_apples}")
    print(f"📊 Average Apples/Image: {batch_summary['average_apples_per_image']}")
    print(f"\n📁 Results saved to:\n{batch_folder}")

📸 Images found in batch: 5

📦 Processing Batch: BATCH_20260823_224048_b6f2c9
🚀 Starting Apple Detection...

✓ 00005.jpg → 1 apple(s) detected
✓ 00006.jpg → 6 apple(s) detected
✓ 00016.jpg → 7 apple(s) detected
✓ 00022.jpg → 4 apple(s) detected
✓ 00032.jpg → 3 apple(s) detected

✅ BATCH PROCESSING COMPLETED!
📦 Batch ID: BATCH_20260823_224048_b6f2c9
📸 Total Images Processed: 5
🍎 Total Apples Detected: 21
📊 Average Apples/Image: 4.2

📁 Results saved to:
C:\major-project\model\batch_results\BATCH_20260823_224048_b6f2c9


: 